# Gurumoji P0 A100 transport smoke

This notebook contains one synthetic job and a small worker source bundle. It does not contain user documents, credentials, or evaluation answers.

Select an A100 GPU runtime, then run this notebook once. It checks a small CUDA matrix operation and runs a deterministic fixture through the Colab worker. It does not call an LLM. The result is still untrusted and is **not accepted** here.

After the result appears, save this notebook to the shared Drive folder (Ctrl+S). Do not rerun it. The result ZIP is stored in the output as base64 so the local acceptance step can verify its hashes and accept the generation once.


In [ ]:
import base64
import hashlib
import json
import sys
import zipfile
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path, PurePosixPath

import torch

BUNDLE_B64 = """..."""
EXPECTED_BUNDLE_SHA256 = "7140aa4636c0bb1a680ca3d4b9e7b1e6b0f4aa7231c81192802f3c5ca8a62a7d"
EXPECTED_JOB_ID = "p0-colab-roundtrip-c11a7321bee8"
EXPECTED_GENERATION = 1

def sha256(data):
    return hashlib.sha256(data).hexdigest()

bundle_bytes = base64.b64decode(BUNDLE_B64, validate=True)
assert sha256(bundle_bytes) == EXPECTED_BUNDLE_SHA256, "P0 bundle checksum mismatch"
with zipfile.ZipFile(BytesIO(bundle_bytes)) as zf:
    expected_members = {"request.zip", "approved_sources.json", "worker_sources.zip", "bundle_manifest.json"}
    assert set(zf.namelist()) == expected_members, "Unexpected bundle files"
    bundle = {name: zf.read(name) for name in expected_members}

bundle_manifest = json.loads(bundle["bundle_manifest.json"])
assert bundle_manifest["job_id"] == EXPECTED_JOB_ID
assert bundle_manifest["generation"] == EXPECTED_GENERATION
for name, digest in bundle_manifest["files"].items():
    assert name in bundle and sha256(bundle[name]) == digest, f"Bundle member hash mismatch: {name}"

# Validate the bundled worker source file set and every source hash before import.
worker_root = Path("/content/p0_worker")
worker_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BytesIO(bundle["worker_sources.zip"])) as zf:
    names = set(zf.namelist())
    assert "worker_sources_manifest.json" in names
    manifest = json.loads(zf.read("worker_sources_manifest.json"))
    source_hashes = manifest["files"]
    assert names == set(source_hashes) | {"worker_sources_manifest.json"}
    for name, digest in source_hashes.items():
        member = PurePosixPath(name)
        assert not member.is_absolute() and ".." not in member.parts
        content = zf.read(name)
        assert sha256(content) == digest, f"Worker source hash mismatch: {name}"
        destination = worker_root.joinpath(*member.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
sys.path.insert(0, str(worker_root / "src"))

from gurumoji.knowledge_builder import contracts, transport
from gurumoji.knowledge_builder.colab_worker import run_colab_worker

registry_value = json.loads(bundle["approved_sources.json"])
assert registry_value["schema_version"] == 1
approved_sources = {source["source_id"]: source for source in registry_value["sources"]}
request_archive, members = transport._read_safe_zip(bundle["request.zip"])
try:
    handoff = transport._parse_canonical_json(transport._zip_read(request_archive, members, "handoff.json"))
    request_data = transport._zip_read(request_archive, members, "request.json")
    request = transport._parse_canonical_json(request_data)
finally:
    request_archive.close()
job = handoff["job"]
assert job["job_id"] == EXPECTED_JOB_ID and job["generation"] == EXPECTED_GENERATION
assert sha256(request_data) == job["input_manifest_sha256"]

# This small CUDA operation confirms the selected A100 is live. The generator
# below is deterministic fixture code, not an LLM call or analytical claim.
assert torch.cuda.is_available(), "CUDA GPU is required; select an A100 runtime"
device_name = torch.cuda.get_device_name(0)
assert "A100" in device_name.upper(), f"Expected A100, found {device_name}"
left = torch.ones((512, 512), device="cuda")
right = torch.ones((512, 512), device="cuda")
product = left @ right
torch.cuda.synchronize()
gpu_first = float(product[0, 0].item())
assert gpu_first == 512.0

def generate(task, excerpts):
    assert task["expert_id"] == "exp-p0-transport-fixture"
    assert len(excerpts) == 1
    excerpt = excerpts[0]
    source = approved_sources[excerpt["source_id"]]
    now = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")
    claim = {
        "schema_version": 1,
        "claim_id": "claim-p0-transport-fixture",
        "expert_id": task["expert_id"],
        "claim": "Synthetic transport fixture only; this is not method guidance.",
        "applicability": ["P0 transport smoke test"],
        "exceptions": [],
        "claim_type": "literature",
        "evidence": [{
            "source_id": source["source_id"], "source_version": source["version"],
            "source_sha256": source["sha256"], "anchor_id": excerpt["anchor_id"],
            "span": {"kind": "page", "start": "1", "end": "1"},
            "extracted_text_sha256": excerpt["extracted_text_sha256"], "transform_history": [],
        }],
        "contradictions": [],
        "status": "candidate",
        "provenance": {"method": "deterministic P0 transport fixture", "created_at": now},
        "rights": source["rights"],
        "reviewer": {"identity": "synthetic-fixture", "reviewed_at": now, "target_sha256": "0" * 64},
    }
    claim["reviewer"]["target_sha256"] = contracts.claim_review_target(claim)
    return [claim]

result_zip = run_colab_worker(bundle["request.zip"], approved_sources=approved_sources, generate=generate)
result_archive, result_members = transport._read_safe_zip(result_zip)
try:
    result_manifest = transport._parse_canonical_json(
        transport._zip_read(result_archive, result_members, transport._RESULT_MANIFEST))
finally:
    result_archive.close()
commit = result_manifest["commit"]
summary = {
    "status": "worker_result_created_unaccepted",
    "job_id": job["job_id"], "generation": job["generation"],
    "request_zip_sha256": sha256(bundle["request.zip"]),
    "input_manifest_sha256": job["input_manifest_sha256"],
    "result_zip_sha256": sha256(result_zip), "result_zip_size_bytes": len(result_zip),
    "commit_manifest_sha256": commit["manifest_sha256"],
    "artifact_count": len(commit["artifacts"]),
    "gpu": device_name, "gpu_matrix_first": gpu_first,
    "generator": "deterministic fixture; no LLM inference",
}
print("P0_RESULT_JSON=" + json.dumps(summary, sort_keys=True))
print("P0_RESULT_ZIP_BASE64=" + base64.b64encode(result_zip).decode("ascii"))
